In [1]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 3.0 MB/s eta 0:00:34
    --------------------------------------- 1.8/101.7 MB 4.4 MB/s eta 0:00:23
   - -------------------------------------- 3.1/101.7 MB 4.5 MB/s eta 0:00:22
   - -------------------------------------- 4.2/101.7 MB 4.8 MB/s eta 0:00:21
   -- ------------------------------------- 5.8/101.7 MB 5.1 MB/s eta 0:00:19
   -- ------------------------------------- 7.1/101.7 MB 5.2 MB/s eta 0:00:19
   --- ------------------------------------ 8.4/101.7 MB 5.2 MB/s eta 0:00:18
   --- ------------------------------------ 9.7/101.7 MB 5.3 MB/s eta 0:00:18
   ---- ----------------------------------- 11.0/101.7 MB 5.3 MB/s eta 0:00:18
   ---- ----------------------------------- 12.3/101.7 MB 5.4 MB/s eta 0:00:17
   ----- ---------------------------------- 13.6/101.7 MB 5.4 MB/s eta 0:00:

In [3]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

# ============================
# 1) تحميل البيانات
# ============================
df = pd.read_csv('data/Hotels_cleaned.csv')

df_model = df.copy()

# إزالة أعمدة التسريب
leakage_cols = ['reservation_status', 'reservation_status_date']
df_model = df_model.drop(columns=leakage_cols, errors='ignore')

# ============================
# 2) Encoding للأعمدة النصية
# ============================
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

# ============================
# 3) فصل البيانات
# ============================
X = df_model.drop('is_canceled', axis=1)
y = df_model['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ============================
# 4) نموذج XGBoost
# ============================
model = XGBClassifier(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'   # أسرع بكثير
)

model.fit(X_train, y_train)

# ============================
# 5) التنبؤ والتقييم
# ============================
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.885082502722171

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.93      0.91     14927
           1       0.87      0.82      0.84      8951

    accuracy                           0.89     23878
   macro avg       0.88      0.87      0.88     23878
weighted avg       0.88      0.89      0.88     23878


Confusion Matrix:
 [[13817  1110]
 [ 1634  7317]]


In [4]:
model.save_model("xgboost_hotel_model.json")
print("Model saved successfully!")


Model saved successfully!


In [5]:
import joblib

joblib.dump(model, "xgboost_hotel_model.pkl")
print("Model saved successfully!")


Model saved successfully!
